<a href="https://colab.research.google.com/github/faizancoder929/Pandas-Data-cleaning/blob/main/E_commerce_Store_Sales_data%20cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np

# A messy sales report export from an online storefront
ecommerce_data = {
    'Order_ID': [1001, 1002, 1003, 1004, 1002, 1005], # Note: 1002 is duplicated!
    'Product_Details': [' T-Shirt - Black - L ', 'hoodie - blue - xl', 'JEANS - INDIGO - M', ' T-Shirt - Black - L ', 'hoodie - blue - xl', 'Sneakers - White - 10'],
    'Units_Sold': [2, np.nan, 1, 2, 2, 5],            # Missing quantity for row 1
    'Item_Cost_USD': [15.0, 35.0, 45.0, 15.0, 35.0, 60.0],
    'Retail_Price_USD': ['$25.00', '   $70.00', 'missing', '$25.00', '$70.00', '$120.00']
}

df = pd.DataFrame(ecommerce_data)
print("--- Raw Shopify Export ---")
print(df)


--- Raw Shopify Export ---
   Order_ID        Product_Details  Units_Sold  Item_Cost_USD Retail_Price_USD
0      1001   T-Shirt - Black - L          2.0           15.0           $25.00
1      1002     hoodie - blue - xl         NaN           35.0           $70.00
2      1003     JEANS - INDIGO - M         1.0           45.0          missing
3      1004   T-Shirt - Black - L          2.0           15.0           $25.00
4      1002     hoodie - blue - xl         2.0           35.0           $70.00
5      1005  Sneakers - White - 10         5.0           60.0          $120.00


In [4]:
# 1. Deduplicate the sales entries
df = df.drop_duplicates()

# 2.Explicitly split and convert the output to text strings immediately
split_data = df['Product_Details'].str.split('-', expand=True)

# Assign the split columns safely
df['Product_Type'] = split_data[0].astype(str).str.strip().str.title()
df['Color'] = split_data[1].astype(str).str.strip().str.title()
df['Size'] = split_data[2].astype(str).str.strip().str.upper()

# 3. Drop the old messy column to clean up the table view
df = df.drop(columns=['Product_Details'])

# 4. Clean the Retail Price column and handle missing values
df['Retail_Price_USD'] = df['Retail_Price_USD'].str.replace('$', '', regex=False).str.strip()
df['Retail_Price_USD'] = pd.to_numeric(df['Retail_Price_USD'], errors='coerce')

# Fill the missing Jeans price based on a standard $90.00 retail valuation rule
df['Retail_Price_USD'] = df['Retail_Price_USD'].fillna(90.0)

# 5. Fix missing units sold by defaulting to a standard single item purchase (1)
df['Units_Sold'] = df['Units_Sold'].fillna(1).astype(int)

# 6. Calculate business performance metrics for the client
df['Total_Revenue'] = df['Retail_Price_USD'] * df['Units_Sold']
df['Total_Cost'] = df['Item_Cost_USD'] * df['Units_Sold']
df['Net_Profit'] = df['Total_Revenue'] - df['Total_Cost']

print("--- Cleaned E-commerce Database ---")
print(df)


--- Cleaned E-commerce Database ---
   Order_ID  Units_Sold  Item_Cost_USD  Retail_Price_USD Product_Type   Color  \
0      1001           2           15.0              25.0            T   Shirt   
1      1002           1           35.0              70.0       Hoodie    Blue   
2      1003           1           45.0              90.0        Jeans  Indigo   
3      1004           2           15.0              25.0            T   Shirt   
4      1002           2           35.0              70.0       Hoodie    Blue   
5      1005           5           60.0             120.0     Sneakers   White   

    Size  Total_Revenue  Total_Cost  Net_Profit  
0  BLACK           50.0        30.0        20.0  
1     XL           70.0        35.0        35.0  
2      M           90.0        45.0        45.0  
3  BLACK           50.0        30.0        20.0  
4     XL          140.0        70.0        70.0  
5     10          600.0       300.0       300.0  


In [5]:
# Calculate total performance metrics using the .sum() function
grand_total_revenue = df['Total_Revenue'].sum()
grand_total_cost = df['Total_Cost'].sum()
net_profit_earned = df['Net_Profit'].sum()
total_items_sold = df['Units_Sold'].sum()

print("=========================================")
print("     KPI SUMMARY FOR STORE OWNER         ")
print("=========================================")
print(f"Total Physical Units Sold : {total_items_sold} items")
print(f"Gross Revenue Generated   : ${grand_total_revenue:,.2f}")
print(f"Total Cost of Goods Sold  : ${grand_total_cost:,.2f}")
print(f"Net Profit Margin Earned  : ${net_profit_earned:,.2f}")
print("=========================================")


     KPI SUMMARY FOR STORE OWNER         
Total Physical Units Sold : 13 items
Gross Revenue Generated   : $1,000.00
Total Cost of Goods Sold  : $510.00
Net Profit Margin Earned  : $490.00
